# Lab 26-06 · How Big, How Organized, Who Pays

Companion to chapter 26-06. Write down each prediction before running the cell.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import team as t

## Part 1 · The flat-team limit

*Predict:* at what size does a flat team with c = 0.03 produce nothing?

In [2]:
c = 0.03
print(f"best size {t.best_flat_size(c):.1f}, zero output at {1 + 1 / c:.1f} people")
n = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(n, np.clip([t.flat_output(x, c) for x in n], 0, None))
ax.set_xlabel("team size"); ax.set_ylabel("output (FTE)")

best size 17.2, zero output at 34.3 people


Text(0, 0.5, 'output (FTE)')

## Part 2 · Splitting 96 engineers

*Predict:* how many teams maximize output?

In [3]:
splits = [k for k in range(1, 97) if 96 % k == 0]
outs = {k: t.hierarchy_output(k, 96 // k, 0.03, 0.02) for k in splits}
for k in splits:
    print(f"{k:3d} teams of {96 // k:2d}: {max(outs[k], 0):6.1f} FTE")
best = max(outs, key=outs.get)
print(f"best: {best} teams, {outs[best]:.1f} FTE")

  1 teams of 96:    0.0 FTE
  2 teams of 48:    0.0 FTE
  3 teams of 32:    6.5 FTE
  4 teams of 24:   28.0 FTE
  6 teams of 16:   47.5 FTE
  8 teams of 12:   55.3 FTE
 12 teams of  8:   59.2 FTE
 16 teams of  6:   57.1 FTE
 24 teams of  4:   47.2 FTE
 32 teams of  3:   34.3 FTE
 48 teams of  2:    5.6 FTE
 96 teams of  1:    0.0 FTE
best: 12 teams, 59.2 FTE


## Part 3 · When hiring stops helping

*Predict:* below how much remaining work do five hires (3-month ramp, half a mentor each) lengthen the project?

In [4]:
diff = lambda W: t.finish_time(W, 10, 0.02) - t.finish_time(W, 10, 0.02, 5, 3, 0.5)
w_be = optimize.brentq(diff, 5, 200)
print(f"break-even remaining work {w_be:.1f} person-months ({w_be / 8.2:.1f} months of current work)")
W = np.linspace(5, 150, 146)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(W, [t.finish_time(x, 10, 0.02) for x in W], label="team of 10")
ax.plot(W, [t.finish_time(x, 10, 0.02, 5, 3, 0.5) for x in W], label="add 5 now")
ax.set_xlabel("remaining work (person-months)"); ax.set_ylabel("months to finish"); ax.legend()

break-even remaining work 51.1 person-months (6.2 months of current work)


## Part 4 · Expensive contractors

*Predict:* the newsvendor base if contractors cost 2.5 times as much as employees.

In [5]:
load = [10, 15, 25, 40, 40, 25, 10, 10]
ce, cc = 75e3, 2.5 * 75e3
base = t.newsvendor_base(load, ce, cc)
print(f"critical ratio {ce / cc:.2f}; base {base:.0f}; cost ${t.staffing_cost(load, base, ce, cc) / 1e6:.1f} M")
for b in range(0, 41, 5):
    print(f"base {b:2d}: ${t.staffing_cost(load, b, ce, cc) / 1e6:5.2f} M")

critical ratio 0.40; base 25; cost $20.6 M
base  0: $32.81 M
base  5: $28.31 M
base 10: $23.81 M
base 15: $22.12 M
base 20: $21.38 M
base 25: $20.62 M
base 30: $21.75 M
base 35: $22.88 M
base 40: $24.00 M
